# 00 — Setup and your first decision
A **decision model** reads a *state* plus questions whose answers are declared in advance, and returns a probability for each answer instead of generated text.

Everything here is **free/open**: public data, open models, open-source libraries. The offline `MockClient` needs no key and no network; a local model is the default backend.


In [ ]:
from decision_qa import load_config, MockClient, SystemOneClient
from decision_qa import NoulQuestion, ChoiceQuestion, ScoreQuestion

cfg = load_config()
print('backend:', cfg.backend, '| endpoint:', cfg.endpoint, '| model:', cfg.model)


## Make a first call
We use `MockClient` so the notebook always runs, for free, even with no model server. Swap in `SystemOneClient(cfg)` when a free local endpoint (Ollama / llama.cpp) is up.


In [ ]:
client = MockClient(seed=42)  # free, deterministic, offline

state = {
    'context': 'A trial of 240 patients reports a 42% reduction in headache frequency.',
    'question': 'Is the treatment effective?',
}
questions = [
    NoulQuestion(label='supported', statement='The treatment is effective.', gold=True),
    ChoiceQuestion(label='verdict', options=['effective', 'ineffective', 'unclear'], gold_index=0),
    ScoreQuestion(label='evidence', rubric='Rate the strength of evidence.',
                  levels=['weak', 'moderate', 'strong'], gold_index=2),
]

for result in client.decision(state, questions):
    print(result.question_label, {k: round(v, 3) for k, v in result.probabilities.items()})


Each result is a **distribution over declared answers**, not a generated sentence. The gold labels above exist only for teaching/evaluation; a real request omits them.


In [ ]:
# Exercise: change the state and watch the probabilities move.
weak_state = {'context': 'A 5-patient anecdote mentions improvement.',
              'question': 'Is the treatment effective?'}
for result in client.decision(weak_state, questions):
    print(result.question_label, {k: round(v, 3) for k, v in result.probabilities.items()})
